In [ ]:
from pathlib import Path
import bcchapi

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
token = (RAIZ / "credenciales.txt").read_text(encoding="utf-8-sig").strip()
siete = bcchapi.Siete(token=token)
siete.buscar("IPSA")

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

busquedas = ["IPSA", "Selectivo", "bursátil", "acciones"]
resultados = pd.concat([siete.buscar(t) for t in busquedas]).drop_duplicates("seriesId")
resultados[["seriesId", "frequencyCode", "spanishTitle", "lastObservation"]]

In [ ]:
print(resultados["frequencyCode"].value_counts())

ipsa = siete.cuadro(
    series=["F013.IBC.IND.N.7.LAC.CL.CLP.BLO.M"],
    nombres=["ipsa_bde"],
    desde="2019-01-01",
    hasta="2026-09-01",
)
ipsa.loc["2023-10-01":"2024-01-01"]

In [ ]:
ruta = RAIZ / "data" / "raw" / "ipsa_investing.csv"
raw = pd.read_csv(ruta, thousands=",")
print(raw.columns.tolist())
ipsa_d = pd.Series(
    raw.iloc[:, 1].astype(float).values,
    index=pd.to_datetime(raw.iloc[:, 0], format="%m/%d/%Y"),
    name="ipsa_investing",
).sort_index()
print(ipsa_d.index.min().date(), "→", ipsa_d.index.max().date(), "|", len(ipsa_d), "días")
ipsa_d.loc["2023-11-28":"2023-12-01"]

In [ ]:
promedio = ipsa_d.resample("MS").mean()
comp = pd.concat([promedio, ipsa["ipsa_bde"]], axis=1, join="inner")
comp["dif_%"] = (comp["ipsa_investing"] / comp["ipsa_bde"] - 1) * 100
print(comp["dif_%"].describe().round(3))
comp.tail(6)